# BulkFormer vs PCA: alternative board layout

Bushra Rashrash · Michigan AI Symposium 2026

This notebook redraws saved aggregate results. It does not fit models or recompute AUROC. Source: `BulkFormer_Poster_Figures_v4_board.ipynb`. The supplied notebook has no saved execution outputs; it has not been run during repository preparation. Plotting code is preserved; outdated explanatory text was corrected.

Use the aggregate CSVs listed in `docs/figure_reproduction.md`. A full participant-level archive is not necessary for plotting. The repository's rounded summary tables do not contain all required inputs.



In [ ]:
from pathlib import Path
import io, json, zipfile, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
try:
    from IPython.display import display
except ImportError:
    display = print

OUTPUT = Path('/content/poster_figures') if Path('/content').exists() else Path.cwd() / 'poster_figures'
OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURE_WIDTH_IN = 12
PNG_DPI = 300
WATERMARK = None          # e.g. 'PREVIEW' for draft renders; None for final figures
ARIAL_TTF = None          # optional path to a licensed Arial .ttf
if ARIAL_TTF:
    font_manager.fontManager.addfont(ARIAL_TTF)
    FONT = font_manager.FontProperties(fname=ARIAL_TTF).get_name()
else:
    installed = {f.name for f in font_manager.fontManager.ttflist}
    FONT = next((f for f in ['Arial', 'Liberation Sans', 'DejaVu Sans'] if f in installed), 'DejaVu Sans')
plt.rcParams.update({
    'font.family': FONT, 'font.size': 24, 'axes.labelsize': 24,
    'axes.titlesize': 24, 'xtick.labelsize': 24, 'ytick.labelsize': 24,
    'legend.fontsize': 24, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'svg.fonttype': 'none', 'savefig.dpi': PNG_DPI, 'figure.dpi': 90,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 1.2, 'axes.titlepad': 16,
})
COLORS = {'PCA': '#003DA5', 'BulkFormer': '#008A91', 'covariates': '#7A828C', 'neutral': '#354150'}
COHORTS = ['GSE251778', 'GSE289146']
TITLES = {'GSE251778': 'MDD\nGSE251778', 'GSE289146': 'Mixed psychiatric\nGSE289146'}
# Analysed class counts (project record, section 2). Checked against each table's n below.
CLASS_COUNTS = {('GSE251778','Female'): (50, 55), ('GSE251778','Male'): (30, 34),
                ('GSE289146','Female'): (135, 46), ('GSE289146','Male'): (94, 24)}
ORIGINAL_SEED = 20260921
exported = []
print('Font:', FONT, '| Export width:', FIGURE_WIDTH_IN, 'inches')


## Load the saved results
Upload the original results ZIP, not old figure files or the historical-reference CSV. Ambiguous matches raise an error instead of choosing silently.

In [ ]:
INPUT_PATHS = []   # outside Colab: a ZIP or list of CSV paths
if INPUT_PATHS:
    uploads = {Path(p).name: Path(p).read_bytes() for p in INPUT_PATHS}
else:
    from google.colab import files
    uploads = files.upload()

csv_files = {}
for filename, content in uploads.items():
    if filename.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            for member in z.namelist():
                if member.lower().endswith('.csv'):
                    csv_files[filename + '::' + member] = pd.read_csv(io.BytesIO(z.read(member)))
    elif filename.lower().endswith('.csv'):
        csv_files[filename] = pd.read_csv(io.BytesIO(content))

def base(name):
    return name.split('::')[-1].split('/')[-1]

def select_table(label, required, preferred_name=None, predicate=None, require_name_or_predicate=False):
    matches = [(n, t) for n, t in csv_files.items()
               if required.issubset(t.columns) and (predicate is None or predicate(t))]
    if preferred_name:
        exact = [(n, t) for n, t in matches if base(n) == preferred_name]
        if exact:
            matches = exact
    if len(matches) != 1:
        raise ValueError(f'{label}: expected one table; found {[m[0] for m in matches]}. '
                         'Upload the results ZIP from one analysis run, or the three specified CSVs.')
    name, table = matches[0]
    print(f'{label}: {name}')
    return table.copy()

repeat_scores = select_table('Repeated metrics', {'cohort','method','seed','AUROC'}, 'repeated_metrics.csv')
extra = [t for n, t in csv_files.items() if base(n) == 'pca_plus_covariates_metrics.csv']
if len(extra) == 1:
    repeat_scores = pd.concat([repeat_scores, extra[0][['cohort', 'method', 'seed', 'AUROC']]], ignore_index=True)
    print('PCA + covariates: pca_plus_covariates_metrics.csv')
original_intervals = select_table('Original bootstrap', {'cohort','comparison','estimate','lower95','upper95'},
    'original_conditional_bootstrap.csv',
    predicate=lambda t: 'sex' not in t.columns and {'PCA','BulkFormer','BF_minus_PCA'}.issubset(set(t.comparison)))

# Separate-sex table: accept it only if its analysis column says separate-sex training,
# or (no analysis column) its file name is exactly separate_sex_conditional_bootstrap.csv.
sex_req = {'cohort','sex','comparison','estimate','lower95','upper95','n'}
sex_candidates = []
for name, t in csv_files.items():
    if not sex_req.issubset(t.columns):
        continue
    if 'analysis' in t.columns:
        if set(t['analysis'].dropna()) == {'separate_sex_training'}:
            sex_candidates.append((name, t))
    elif base(name) == 'separate_sex_conditional_bootstrap.csv':
        sex_candidates.append((name, t))
if len(sex_candidates) != 1:
    raise ValueError('Separate-sex bootstrap: expected exactly one table labelled separate_sex_training '
                     f'or named separate_sex_conditional_bootstrap.csv; found {[c[0] for c in sex_candidates]}. '
                     'The pooled-model subgroup table is a different analysis and must not be used here.')
print('Separate-sex bootstrap:', sex_candidates[0][0])
sex_intervals = sex_candidates[0][1].copy()

needed = ['PCA','full131','learned128','summaries3','age_BMI_sex','BF_plus_age_BMI_sex']
for cohort in COHORTS:
    for method in needed:
        part = repeat_scores.query('cohort == @cohort and method == @method')
        if len(part) != 5 or part.seed.nunique() != 5:
            raise ValueError(f'{cohort}/{method}: expected five unique repeat seeds, found {len(part)} rows.')
        if not np.isfinite(part.AUROC).all() or not part.AUROC.between(0,1).all():
            raise ValueError(f'Invalid AUROC values for {cohort}/{method}.')
    seeds = [set(repeat_scores.query('cohort == @cohort and method == @m').seed) for m in needed]
    if any(s != seeds[0] for s in seeds[1:]):
        raise ValueError(f'Unmatched repeat seeds in {cohort}.')
for table in [original_intervals, sex_intervals]:
    if not np.isfinite(table[['estimate','lower95','upper95']].to_numpy()).all():
        raise ValueError('Nonfinite interval values.')
    if not ((table.lower95 <= table.estimate) & (table.estimate <= table.upper95)).all():
        raise ValueError('An estimate lies outside its interval.')
for (cohort, sex), (cases, controls) in CLASS_COUNTS.items():
    n = sex_intervals.query('cohort == @cohort and sex == @sex')['n'].unique()
    if len(n) != 1 or int(n[0]) != cases + controls:
        raise ValueError(f'{cohort}/{sex}: table n={n} does not equal recorded {cases}+{controls}. Update CLASS_COUNTS from the analysis notebook.')

repeat_summary = repeat_scores.groupby(['cohort','method']).AUROC.agg(['mean','std','min','max','count'])
display(repeat_summary.round(4))
repeat_summary.to_csv(OUTPUT / 'plotted_repeat_summary.csv')

# Provenance: original split vs repeat seed 20260921 (expected to share partitions).
rows = []
for cohort in COHORTS:
    for comp, method in [('PCA','PCA'), ('BulkFormer','full131')]:
        o = original_intervals.query('cohort == @cohort and comparison == @comp').estimate
        r = repeat_scores.query('cohort == @cohort and method == @method and seed == @ORIGINAL_SEED').AUROC
        if len(o) == 1 and len(r) == 1:
            rows.append({'cohort': cohort, 'method': comp, 'original_split': float(o.iloc[0]),
                         'repeat_seed_20260921': float(r.iloc[0]), 'abs_diff': abs(float(o.iloc[0]) - float(r.iloc[0]))})
prov = pd.DataFrame(rows)
display(prov.round(5))
prov.to_csv(OUTPUT / 'original_vs_repeat1_check.csv', index=False)
if (prov.abs_diff > 0.0005).any():
    warnings.warn('Original-split and repeat-1 estimates differ by >0.0005; record why before reusing either.')


## Plot helpers
0–1 AUROC scale (or symmetric zero-centred scale for differences). Blue and teal are reserved for PCA and BulkFormer; gray marks covariates-only and means. Fixed figure size; no tight cropping.

In [ ]:
def style_axis(ax):
    ax.set_axisbelow(True)
    ax.grid(axis='y', color='#E4E8EC', linewidth=0.9)
    ax.tick_params(length=5, pad=9)

def save_plot(fig, name):
    if WATERMARK:
        fig.text(0.5, 0.5, WATERMARK, ha='center', va='center', fontsize=40, color='#C0392B',
                 alpha=0.22, rotation=18, wrap=True)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for text in fig.findobj(matplotlib.text.Text):
        if text.get_visible() and text.get_text():
            b = text.get_window_extent(renderer)
            if b.x0 < -2 or b.y0 < -2 or b.x1 > fig.bbox.width + 2 or b.y1 > fig.bbox.height + 2:
                raise RuntimeError(f'Text extends outside {name}: {text.get_text()!r}. Increase figure height/margins.')
    for ext in ['png', 'pdf', 'svg']:
        path = OUTPUT / f'{name}.{ext}'
        fig.savefig(path, dpi=PNG_DPI, facecolor='white', bbox_inches=None)
        exported.append(path.name)
    plt.show()
    plt.close(fig)

def two_panel(height=6.6, bottom=.27):
    fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_IN, height), sharey=True)
    fig.subplots_adjust(left=.14, right=.975, bottom=bottom, top=.79, wspace=.20)
    for ax, cohort in zip(axes, COHORTS):
        ax.set_title(TITLES[cohort])
        ax.set_ylim(0, 1)
        ax.set_yticks(np.linspace(0, 1, 6))
        ax.axhline(.5, color='#A7AEB6', ls=':', lw=1.5)
        style_axis(ax)
    axes[0].set_ylabel('Held-out AUROC', labelpad=12)
    return fig, axes


## 1. Main comparison — five matched repeats
Each line is one matched repeat; repeats reuse the same participants. Means are printed under the labels (descriptive, not confidence intervals).

In [ ]:
fig, axes = two_panel(5.6, bottom=.30)
for ax, cohort in zip(axes, COHORTS):
    w = repeat_scores.query('cohort == @cohort').pivot(index='seed', columns='method', values='AUROC')
    for _, row in w.iterrows():
        ax.plot([0, 1], [row['PCA'], row['full131']], color='#AEBAC7', lw=2, zorder=2)
        ax.scatter([0, 1], [row['PCA'], row['full131']], c=[COLORS['PCA'], COLORS['BulkFormer']], s=110, zorder=3)
    ax.set_xlim(-.30, 1.30)
    ax.set_xticks([0, 1], [f"PCA\nmean {w['PCA'].mean():.3f}", f"BulkFormer\nmean {w['full131'].mean():.3f}"])
save_plot(fig, 'paired_repeats')


## 2. Original-split paired differences
Conditional 95% paired bootstrap intervals from fixed out-of-fold predictions of the original split; retraining uncertainty excluded. Not attached to repeat means.

In [ ]:
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, 4.0))
fig.subplots_adjust(left=.30, right=.68, bottom=.28, top=.95)
rows = []
for i, cohort in enumerate(COHORTS):
    part = original_intervals.query('cohort == @cohort and comparison == "BF_minus_PCA"')
    if len(part) != 1:
        raise ValueError('Expected one paired difference per cohort.')
    r = part.iloc[0]; rows.append(r)
    ax.hlines(i, r.lower95, r.upper95, color=COLORS['neutral'], lw=3.5)
    ax.scatter(r.estimate, i, color=COLORS['neutral'], s=130, zorder=3)
limit = max(.10, np.ceil(max(max(abs(r.lower95), abs(r.upper95)) for r in rows) / .05) * .05)
for i, r in enumerate(rows):
    fmt = lambda x: f'{x:+.3f}'.replace('-', '−')
    ax.text(limit * 1.06, i, f'{fmt(r.estimate)}\n[{fmt(r.lower95)}, {fmt(r.upper95)}]',
            va='center', ha='left', fontsize=24, transform=ax.transData, clip_on=False)
ax.set_xlim(-limit, limit)
ax.set_xticks([-0.1, 0, 0.1], ['−0.1', '0', '+0.1'])
ax.axvline(0, color='#606975', ls='--', lw=1.7)
ax.set_yticks([0, 1], [TITLES[c] for c in COHORTS])
ax.set_ylim(1.6, -.6)
ax.set_xlabel('AUROC difference (BulkFormer − PCA)', labelpad=14)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0, pad=16)
ax.grid(axis='x', color='#E4E8EC')
save_plot(fig, 'original_paired_difference')


## 3. Feature and covariate sensitivity
Dots: five repeat scores (horizontal offsets are for visibility only). Dark bars: repeat means, not confidence intervals. Covariates = age, BMI and recorded sex. Dashed line in the covariate figure: PCA repeat mean for reference. The separate covariate panel shows BulkFormer-based comparisons. PCA + covariates was evaluated in the final analysis and is included in the combined panel when its saved metrics are supplied.

In [ ]:
families = [
    ('feature_ablation', ['full131','learned128','summaries3'],
     ['Full\n131 features','Learned\n128 features','Summary\n3 features'], False),
    ('covariates', ['full131','age_BMI_sex','BF_plus_age_BMI_sex'],
     ['BulkFormer','Covariates\nonly','BulkFormer\n+ covariates'], True),
]
for name, methods, labels, show_pca in families:
    fig, axes = two_panel(7.3, bottom=.33)
    for ax, cohort in zip(axes, COHORTS):
        for j, method in enumerate(methods):
            values = repeat_scores.query('cohort == @cohort and method == @method').sort_values('seed').AUROC.to_numpy()
            colour = COLORS['covariates'] if method == 'age_BMI_sex' else COLORS['BulkFormer']
            ax.scatter(j + np.linspace(-.10, .10, len(values)), values, color=colour, s=85, zorder=3)
            ax.hlines(values.mean(), j - .23, j + .23, color=COLORS['neutral'], lw=3.5, zorder=4)
        if show_pca:
            pca_mean = repeat_scores.query('cohort == @cohort and method == "PCA"').AUROC.mean()
            ax.axhline(pca_mean, color=COLORS['PCA'], ls='--', lw=2.2, zorder=1)
        ax.set_xlim(-.55, 2.55)
        ax.set_xticks(range(3), labels)
        ax.tick_params(axis='x', labelsize=24)
    handles = [Line2D([], [], marker='o', ls='', color=COLORS['BulkFormer'], markersize=10, label='BulkFormer repeat'),
               Line2D([], [], color=COLORS['neutral'], lw=3, label='Mean')]
    if show_pca:
        handles.insert(1, Line2D([], [], marker='o', ls='', color=COLORS['covariates'], markersize=10, label='Covariates only'))
        handles.append(Line2D([], [], color=COLORS['PCA'], ls='--', lw=2.2, label='PCA mean'))
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(.54, .005), ncol=len(handles),
               frameon=False, fontsize=22, columnspacing=1.2, handletextpad=.4)
    save_plot(fig, name)


## 3b. Combined sensitivity figure (recommended for the poster)
One figure replaces the two above: five feature sets per cohort on a shared vertical list, so long labels stay horizontal at 24 pt. Dots: five repeats; dark ticks: repeat means (not confidence intervals); dashed blue line: PCA repeat mean. The combined panel includes PCA + covariates when pca_plus_covariates_metrics.csv is supplied.

In [ ]:
SENS = [('full131', 'BulkFormer (131)'), ('learned128', 'Learned only (128)'),
        ('summaries3', 'Summaries only (3)'), ('age_BMI_sex', 'Age, BMI & sex'),
        ('BF_plus_age_BMI_sex', 'BulkFormer + covariates'), ('PCA_plus_age_BMI_sex', 'PCA + covariates')]
SENS = [(m, l) for m, l in SENS if m in set(repeat_scores.method)]
fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_IN, 6.6 + 0.5 * (len(SENS) - 5)), sharey=True)
fig.subplots_adjust(left=.375, right=.975, bottom=.34, top=.87, wspace=.22)
for ax, cohort in zip(axes, COHORTS):
    for i, (method, _) in enumerate(SENS):
        values = repeat_scores.query('cohort == @cohort and method == @method').sort_values('seed').AUROC.to_numpy()
        colour = COLORS['covariates'] if method == 'age_BMI_sex' else COLORS['PCA'] if method.startswith('PCA') else COLORS['BulkFormer']
        ax.scatter(values, i + np.linspace(-.16, .16, len(values)), color=colour, s=80, zorder=3)
        ax.vlines(values.mean(), i - .34, i + .34, color=COLORS['neutral'], lw=4, zorder=4)
    pca_mean = repeat_scores.query('cohort == @cohort and method == "PCA"').AUROC.mean()
    ax.axvline(pca_mean, color=COLORS['PCA'], ls='--', lw=2.4, zorder=1)
    ax.axvline(.5, color='#A7AEB6', ls=':', lw=1.5, zorder=0)
    ax.set_xlim(0, 1); ax.set_xticks([0, .5, 1], ['0', '0.5', '1'])
    ax.set_title(TITLES[cohort], fontsize=24)
    ax.tick_params(axis='y', length=0)
    ax.set_axisbelow(True); ax.grid(axis='x', color='#E4E8EC'); ax.tick_params(length=5, pad=8)
    ax.spines['left'].set_visible(False)
axes[0].set_yticks(range(len(SENS)), [label for _, label in SENS])
axes[0].tick_params(axis='y', labelsize=24)
for ax in axes:
    ax.tick_params(axis='y', length=0)
axes[0].set_ylim(len(SENS) - .45, -.55)
fig.supxlabel("Held-out AUROC", x=.675, y=.185, fontsize=24)
handles = [Line2D([], [], marker='o', ls='', color=COLORS['BulkFormer'], markersize=10, label='BulkFormer-based'),
           Line2D([], [], marker='o', ls='', color=COLORS['PCA'], markersize=10, label='PCA-based'),
           Line2D([], [], marker='o', ls='', color=COLORS['covariates'], markersize=10, label='Covariates only'),
           Line2D([], [], color=COLORS['neutral'], lw=4, label='Mean of 5 repeats'),
           Line2D([], [], color=COLORS['PCA'], ls='--', lw=2.4, label='PCA alone, mean')]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(.5, .0), ncol=3, frameon=False,
           fontsize=24, columnspacing=.9, handletextpad=.3)
save_plot(fig, 'sensitivity_combined')


## 4. Exploratory models trained separately by recorded sex
Separate-sex training (four outer, three inner folds, one seed), not subgroup evaluation of the pooled model. In GSE251778, recorded sex coincides with library kit and sequencer, so sex and protocol cannot be separated.

In [ ]:
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, 8.0))
fig.subplots_adjust(left=.43, right=.97, bottom=.165, top=.88)
labels = []
for i, (cohort, sex) in enumerate((c, s) for c in COHORTS for s in ['Female', 'Male']):
    part = sex_intervals.query('cohort == @cohort and sex == @sex')
    cases, controls = CLASS_COUNTS[(cohort, sex)]
    case_word = 'MDD' if cohort == 'GSE251778' else 'affected'
    ctrl_word = 'controls' if cohort == 'GSE251778' else 'unaffected'
    cohort_label = 'MDD' if cohort == 'GSE251778' else 'Mixed psychiatric'
    labels.append(f'{cohort_label} · {sex}\n{cases} {case_word} / {controls} {ctrl_word}')
    for method, offset in [('PCA', -.13), ('BulkFormer', .13)]:
        selected = part.loc[part.comparison == method]
        if len(selected) != 1:
            raise ValueError(f'Expected one {cohort}/{sex}/{method} row.')
        r = selected.iloc[0]
        ax.hlines(i + offset, r.lower95, r.upper95, color=COLORS[method], lw=3)
        ax.scatter(r.estimate, i + offset, color=COLORS[method], s=95, label=method if i == 0 else None, zorder=3)
ax.set_yticks(range(4), labels)
ax.tick_params(axis='y', labelsize=24, length=0, pad=14)
ax.set_ylim(3.6, -.6)
ax.set_xlim(0, 1); ax.set_xticks(np.linspace(0, 1, 6))
ax.axvline(.5, color='#A7AEB6', ls=':', lw=1.5)
ax.axhline(1.5, color='#D5DAE0', lw=1.2)
ax.set_xlabel('Held-out AUROC (95% interval)', labelpad=12)
ax.spines['left'].set_visible(False)
ax.grid(axis='x', color='#E4E8EC')
ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.02), ncol=2, frameon=False)
save_plot(fig, 'separate_sex_intervals')


## Export

The code saves PNGs at 300 dpi and vector PDFs. Actual canvas sizes are set in each plotting cell; preserve aspect ratio when placing them. Font size changes with scaling. These are figure canvases, not the overall poster page dimensions.


In [ ]:
settings = {'font': FONT, 'width_inches': FIGURE_WIDTH_IN, 'png_dpi': PNG_DPI,
            'tick_font_pt': 26, 'category_tick_font_pt': 22, 'axis_label_font_pt': 28,
            'colours': COLORS, 'results_recomputed': False, 'watermark': WATERMARK,
            'intervals': 'Conditional on fixed predictions; exclude retraining uncertainty',
            'repeat_spread': 'Descriptive, not confidence intervals'}
(OUTPUT / 'figure_settings.json').write_text(json.dumps(settings, indent=2))
(OUTPUT / 'README.txt').write_text(
    'Insert PNGs at 14 inch width and lock aspect ratio. PDF/SVG are vector alternatives.\n'
    'These are redraws of uploaded results, not newly fitted models.\n'
    'Conditional bootstrap intervals are not repeated-CV uncertainty intervals.\n'
    'Repeated observations reuse participants. Sex models are exploratory.\n'
    'The mixed psychiatric cohort is not an MDD-only validation cohort.\n')
archive = shutil.make_archive(str(OUTPUT.parent / 'BulkFormer_Poster_Figures'), 'zip', OUTPUT)
print('Saved:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Open the ZIP at the path above.')
